In [1]:
%reload_ext autoreload
from data_generation.network_generator import StochasticBlockModel
from data_generation.data_simulator import IndividualDataSimulator
from constants import NetworkSettings, DataGenerationSettings

/Users/polinarevina/Library/Caches/pypoetry/virtualenvs/simulations-notebooks-2NAxyaa6-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/polinarevina/Library/Caches/pypoetry/virtualenvs/simulations-notebooks-2NAxyaa6-py3.12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [2]:
network_settings = NetworkSettings()
dgp_settings = DataGenerationSettings()

In [5]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

def corrupt_network(graph, removal_prob=0.2, addition_prob=0.1):
    """
    Вносит ошибки в сеть:
    - Случайно удаляет часть связей с вероятностью `removal_prob`
    - Добавляет случайные связи с вероятностью `addition_prob`
    Возвращает:
    - corrupted_graph: networkx.Graph — искажённый граф
    """
    corrupted_graph = graph.copy()
    edges = list(corrupted_graph.edges())
    nodes = list(corrupted_graph.nodes())
    
    # Удаление случайных связей
    num_remove = int(len(edges) * removal_prob)
    edges_to_remove = np.random.choice(len(edges), num_remove, replace=False)
    for idx in edges_to_remove:
        corrupted_graph.remove_edge(*edges[idx])
    
    # Добавление случайных связей
    num_add = int(len(edges) * addition_prob)
    for _ in range(num_add):
        u, v = np.random.choice(nodes, 2, replace=False)
        if not corrupted_graph.has_edge(u, v):  # Добавляем, если связи не было
            corrupted_graph.add_edge(u, v)
    
    return corrupted_graph

In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Генерация графа
import networkx as nx
G = nx.barabasi_albert_graph(100, 3)  # Случайная сеть
edge_index = torch.tensor(list(G.edges())).t().contiguous()

# Данные
X = torch.tensor(df[['treatment', 'exposure_fraction', 'age', 'income']].values, dtype=torch.float)
y = torch.tensor(df['Y'].values, dtype=torch.float)

# GNN-модель
class GNN(torch.nn.Module):
    def __init__(self):
        super(GNN, self).__init__()
        self.conv1 = SAGEConv(X.shape[1], 16)
        self.conv2 = SAGEConv(16, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x.view(-1)

# Обучение модели
model = GNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()
    pred = model(X, edge_index)
    loss = F.mse_loss(pred, y)
    loss.backward()
    optimizer.step()

print("GNN Loss:", loss.item())
